# CSIRO Competition Solution Notebook

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth
/kaggle/input/dinov2/pytorch/giant/1/config.json
/kaggle/input/dinov2/pytorch/giant/1/preprocessor_config.json
/kaggle/input/dinov2/pytorch/giant/1/README.md
/kaggle/input/dinov2/pytorch/giant/1/pytorch_model.bin
/kaggle/input/dinov2/pytorch/giant/1/.gitattributes
/kaggle/input/dinov2/pytorch/base/1/config.json
/kaggle/input/dinov2/pytorch/base/1/preprocessor_config.json
/kaggle/input/dinov2/pytorch/base/1/README.md
/kaggle/input/dinov2/pytorch/base/1/pytorch_model.bin
/kaggle/input/dinov2/pytorch/base/1/.gitattributes
/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID2099464826.jpg
/kaggle/input/csiro-biomass/train/ID2037861084.jpg
/kaggle/input/csiro-biomass/train/ID1211362607.jpg
/kaggle/input/csiro-biomass/train/ID1853508321.jpg
/k

In [2]:
import shutil
import os

# Copy entire dataset folder
input_folder = "/kaggle/input/csiro-biomass"
output_folder = "/kaggle/working/csiro-biomass"

# Copy entire directory
shutil.copytree(input_folder, output_folder)

print(f"✓ Folder copied to: {output_folder}")

# Update paths
dataset_path = "/kaggle/working/csiro-biomass/train.csv"
print(f"dataset_path = '{dataset_path}'")

# List copied files
print(f"\nCopied files:")
for item in os.listdir(output_folder):
    item_path = os.path.join(output_folder, item)
    if os.path.isfile(item_path):
        size = os.path.getsize(item_path) / (1024 * 1024)
        print(f"  {item}: {size:.2f} MB")
    else:
        num_files = len(os.listdir(item_path))
        print(f"  {item}/: {num_files} files")

✓ Folder copied to: /kaggle/working/csiro-biomass
dataset_path = '/kaggle/working/csiro-biomass/train.csv'

Copied files:
  train/: 357 files
  train.csv: 0.17 MB
  sample_submission.csv: 0.00 MB
  test.csv: 0.00 MB
  test/: 1 files


In [3]:
import sys
sys.path.append("/kaggle/input/sam-optim")

In [4]:
import torch

torch.cuda.is_available()

False

In [5]:
from sam import *

# Data Cleaning

## Inconsistent Total & GDM calculation

In [6]:
import pandas as pd
import numpy as np

# Load, clean, and save
dataset_path = "/kaggle/working/csiro-biomass/train.csv"
output_path = dataset_path

df = pd.read_csv(dataset_path)
id_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']

# Pivot and check consistency
df_wide = df.pivot_table(index=id_cols, columns='target_name', values='target').reset_index()
df_wide['GDM_diff'] = abs(df_wide['GDM_g'] - (df_wide['Dry_Clover_g'] + df_wide['Dry_Green_g']))
df_wide['Total_diff'] = abs(df_wide['Dry_Total_g'] - (df_wide['Dry_Clover_g'] + df_wide['Dry_Green_g'] + df_wide['Dry_Dead_g']))

# Get bad images and remove
tolerance = 0.01
bad_images = df_wide[(df_wide['GDM_diff'] > tolerance) | (df_wide['Total_diff'] > tolerance)]['image_path'].unique()
df_clean = df[~df['image_path'].isin(bad_images)]

# Save and update path
df_clean.to_csv(output_path, index=False)
dataset_path = output_path

print(f"Removed {len(bad_images)} inconsistent images")
print(f"Cleaned data: {df_clean.shape[0]} rows, {df_clean['image_path'].nunique()} images")
print(f"Saved to: {dataset_path}")

Removed 1 inconsistent images
Cleaned data: 1780 rows, 356 images
Saved to: /kaggle/working/csiro-biomass/train.csv


# Data Augmentation & Transform

In [7]:
# Data Transform

from torchvision.transforms import v2
import torch

# to_tensor = v2.ToTensor()
# img_tensor = to_tensor(img)

dtype = torch.float32
img_size = (224, 224)
image_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype, scale=True),    
    v2.Resize(img_size),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(5, interpolation=v2.InterpolationMode.BILINEAR),
    v2.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.25,
        hue=0.05,
    ),
    # v2.RandomAdjustSharps
    v2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]),
])

val_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype, scale=True),
    v2.Resize(img_size),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def numeric_transform(X, X_max, X_min) -> torch.Tensor:
    X_normalized = (X - X_min) / (X_max - X_min)
    return X_normalized

def target_transform(targets) -> torch.Tensor:
    return torch.log1p(targets)

def target_untransform(targets) -> torch.Tensor:
    return torch.expm1(targets)

def categorical_transform(row) -> torch.Tensor:
    return row

# Train Set

In [8]:

from torch.utils.data import Dataset
from torchvision.io import decode_image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pandas as pd

class Image2BioMassTrainValDataset(Dataset):
    
    def __init__(self, dataset_path, img_transform=None, numeric_transform=None,categorical_transform=None, target_transform=None):
        
        self.df = self.process_df(dataset_path)
        self.dataset_path = dataset_path
        self.img_transform = img_transform
        self.target_transform = target_transform
        self.numeric_transform = numeric_transform
        self.categorical_transform = categorical_transform
        self.targets = self.df.loc[:, ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g"]]


    def process_df(self, dataset_path):
        self.le_date = LabelEncoder()
        self.le_state = LabelEncoder()
        self.le_species = LabelEncoder()

        df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
        df['base_sample_id'] = df['sample_id'].str.split('__').str[0]
        df = df.pivot_table(
        index=['base_sample_id', 'image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'],
        columns='target_name',
        values='target'
        ).reset_index()
        df["Sampling_Date"] = self.le_date.fit_transform(df["Sampling_Date"])
        df["State"] = self.le_state.fit_transform(df["State"])
        df["Species"] = self.le_species.fit_transform(df["Species"])
        # display(df)
        return df

    def __len__(self):
        return len(self.df)

    def get_cat_features(self):
        return ["Sampling_Date", "State", "Species"]
    
    def get_cat_vocab_sizes(self):
        results = []

        for i in self.get_cat_features():
            results.append(len(self.df[i].unique()))
        return results

    def __getitem__(self, idx):
        # B = batch_size
        # display(self.df)
        img_path = os.path.join(self.dataset_path, self.df.loc[idx, 'image_path'])
        image = decode_image(img_path)
        # display(self.df)
        numeric_features = torch.tensor([
            self.df.loc[idx, "Pre_GSHH_NDVI"],
            self.df.loc[idx, "Height_Ave_cm"],
        ], dtype=torch.float32)

        categorical_features = torch.tensor([
            self.df.loc[idx, "Sampling_Date"],
            self.df.loc[idx, "State"],
            self.df.loc[idx, "Species"],
        ], dtype=torch.long)
        

        if self.img_transform:
            image = self.img_transform(image)
            
        if self.numeric_transform:
            # numeric_features[0] = self.numeric_transform(
            #     numeric_features[0],
            #     self.df.loc[:, "Pre_GSHH_NDVI"].max(), 
            #     self.df.loc[:, "Pre_GSHH_NDVI"].min()
            # )
            numeric_features[1] = self.numeric_transform(
                numeric_features[1], 
                self.df.loc[:, "Height_Ave_cm"].max(), 
                self.df.loc[:, "Height_Ave_cm"].min()
            )
            # print(numeric_features)
        combined_features = torch.cat([categorical_features.float(), numeric_features], dim=0)
        # print(combined_features)
        targets = torch.Tensor(self.targets.iloc[idx].values)
        if self.target_transform:
            targets = self.target_transform(targets)
        return image, combined_features, targets

# Test Set

In [9]:

# from torch.utils.data import Dataset
# from torchvision.io import decode_image
# from sklearn.preprocessing import LabelEncoder
# from sklearn.model_selection import train_test_split
# from torch.utils.data import DataLoader
# import pandas as pd

# class Image2BioMassTestFromTrainDataset(Dataset):
    
#     def __init__(self, dataset_path, img_transform=None, numeric_transform=None,categorical_transform=None):
        
#         self.df = self.process_df(dataset_path)
#         self.dataset_path = dataset_path
#         self.img_transform = img_transform
#         self.numeric_transform = numeric_transform
#         self.categorical_transform = categorical_transform

#     def process_df(self, dataset_path):
#         self.le_date = LabelEncoder()
#         self.le_state = LabelEncoder()
#         self.le_species = LabelEncoder()

#         df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
#         df['base_sample_id'] = df['sample_id'].str.split('__').str[0]
#         df = (
#             df.assign(_val="")
#               .pivot(index=['base_sample_id', "image_path"],
#                      columns='target_name',
#                      values='_val')
#               .reset_index()
#         )

#         return df

#     def __len__(self):
#         return len(self.df)

#     def get_cat_features(self):
#         return ["Sampling_Date", "State", "Species"]
    
#     def get_cat_vocab_sizes(self):
#         results = []

#         for i in self.get_cat_features():
#             results.append(len(self.df[i].unique()))
#         return results

#     def __getitem__(self, idx):

#         img_path = os.path.join(self.dataset_path, self.df.loc[idx, 'image_path'])
#         image = decode_image(img_path)

#         # Use val_transform for test data (no augmentation)
#         if self.img_transform:
#             image = self.img_transform(image)
#         else:
#             # Fallback basic transform if no transform provided
#             transform = v2.Compose([
#                 v2.ToImage(),
#                 v2.ToDtype(dtype, scale=True),
#                 v2.Resize((518, 518)),
#                 v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             ])
#             image = transform(image)

#         combined_features = torch.zeros(5, dtype=torch.float32)
#         sample_id = self.df.loc[idx, 'base_sample_id']
#         return image, combined_features, sample_id

In [10]:

from torch.utils.data import Dataset
from torchvision.io import decode_image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pandas as pd

class Image2BioMassTestDataset(Dataset):
    
    def __init__(self, dataset_path, img_transform=None, numeric_transform=None,categorical_transform=None):
        
        self.df = self.process_df(dataset_path)
        self.dataset_path = dataset_path
        self.img_transform = img_transform
        self.numeric_transform = numeric_transform
        self.categorical_transform = categorical_transform

    def process_df(self, dataset_path):
        self.le_date = LabelEncoder()
        self.le_state = LabelEncoder()
        self.le_species = LabelEncoder()

        df = pd.read_csv(os.path.join(dataset_path, "test.csv"))
        df['base_sample_id'] = df['sample_id'].str.split('__').str[0]
        df = (
            df.assign(_val="")
              .pivot(index=['base_sample_id', "image_path"],
                     columns='target_name',
                     values='_val')
              .reset_index()
        )

        return df

    def __len__(self):
        return len(self.df)

    def get_cat_features(self):
        return ["Sampling_Date", "State", "Species"]
    
    def get_cat_vocab_sizes(self):
        results = []

        for i in self.get_cat_features():
            results.append(len(self.df[i].unique()))
        return results

    def __getitem__(self, idx):

        img_path = os.path.join(self.dataset_path, self.df.loc[idx, 'image_path'])
        image = decode_image(img_path)

        # Use val_transform for test data (no augmentation)
        if self.img_transform:
            image = self.img_transform(image)
        else:
            # Fallback basic transform if no transform provided
            transform = v2.Compose([
                v2.ToImage(),
                v2.ToDtype(dtype, scale=True),
                v2.Resize((518, 518)),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
            image = transform(image)

        combined_features = torch.zeros(5, dtype=torch.float32)
        sample_id = self.df.loc[idx, 'base_sample_id']
        return image, combined_features, sample_id

In [11]:
test_dataset = Image2BioMassTestDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=val_transform,  # Use val_transform (no augmentation, proper size)
    
)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)
next(iter(test_dataloader))[2]

('ID1001187975',)

# Train Split

In [12]:
import torch, random, numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

g = torch.Generator()
g.manual_seed(42)

In [13]:

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

# Create base dataset to get indices
base_dataset = Image2BioMassTrainValDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=None,  # no transform yet
    numeric_transform=numeric_transform,
    target_transform=target_transform
)

# Split indices
seed = 42
train_indices, val_indices = train_test_split(
    range(len(base_dataset)), 
    train_size=0.8, 
    shuffle=True, 
    random_state=seed
)

# Create training dataset WITH augmentation
train_dataset = Image2BioMassTrainValDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=image_transform,  # WITH augmentation
    numeric_transform=numeric_transform,
    target_transform=target_transform
)
train_dataset = Subset(train_dataset, train_indices)

val_dataset = Image2BioMassTrainValDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=val_transform,  # WITHOUT augmentation
    numeric_transform=numeric_transform,
    target_transform=target_transform
)
val_dataset = Subset(val_dataset, val_indices)

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True, generator=g)
val_dataloader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# Model

In [14]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision.models import resnet152, ResNet152_Weights


RESNET_PATH = "/kaggle/input/resnet50/pytorch/default/1/resnet50-0676ba61.pth"
class BackBone(nn.Module):

    def __init__(self):

        
        super().__init__()
        pass

    def forward(self, x):
        pass

class Image2BiomassModel(nn.Module):

    def __init__(self):
        super().__init__()

        # ---- load DINOv2 giant backbone from local ----
        # from transformers import Dinov2Model
        # self.backbone = Dinov2Model.from_pretrained(
        #     "/kaggle/input/dinov2/pytorch/giant/1/"
        # )

        # self.backbone = BackBone()
        backbone = resnet152(weights=ResNet152_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])

        for param in self.backbone.parameters():
            param.requires_grad = False
        

        self.noise = nn.Sequential(
            nn.AlphaDropout(0.1),
        )
        # DINOv2-giant outputs 1536-dim features
        self.fc1 = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.Mish(),
            nn.Dropout(0.4),
        )
        # self.fc1 = nn.Sequential(
        #     nn.Linear(1536, 1024),
        #     nn.BatchNorm1d(1024),
        #     nn.Mish(),
        #     nn.Dropout(0.4),
        # )

        self.fc2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.Mish(),
            nn.Dropout(0.4),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.Mish(),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
        )

        self.out = nn.Linear(512, 3)

        self.criterion = nn.SmoothL1Loss(beta=0.5)

    def forward(self, x, y=None):
        # DINOv2-giant expects normalized images and outputs [B, 1536]
        # outputs = self.backbone(x)
        x = self.backbone(x)
        # x = outputs.last_hidden_state[:, 0]  # Take [CLS] token
        x = x.view(x.size(0), -1)
        x = self.noise(x)
        x = self.fc1(x)
        x = self.fc2(x)
        preds = self.out(x)

        loss = None
        if y is not None:
            loss = self.criterion(preds, y)

        return preds, loss


# sample = next(iter(train_dataloader))
# model = Image2BiomassModel()

# model(sample[0], sample[2])

In [15]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [16]:
# import torch
# from torch import nn
# import torch.nn.functional as F

# BATCH_SIZE=32
# HEIGHT=224
# WIDTH=224
# NUM_CHANNELS=3
# class BackBone(nn.Module):

#     def __init__(self, num_channels=3):
#         super(BackBone, self).__init__()

#         self.conv1 = nn.Conv2d(in_channels=num_channels, out_channels=32, kernel_size=3, padding=1)
#         self.batch_norm1 = nn.BatchNorm2d(32)
#         self.activ1 = nn.GELU()
#         self.conv2 = nn.Conv2d(in_channels=32, out_channels=16, kernel_size=3, padding=1)
#         self.batch_norm2 = nn.BatchNorm2d(16)
#         self.activ2 = nn.GELU()
#         self.conv3 = nn.Conv2d(in_channels=16, out_channels=3, kernel_size=3, padding=1)
#         self.batch_norm3 = nn.BatchNorm2d(3)
#         self.activ3 = nn.GELU()

#         self.downsample = None

#     def forward(self, x):
#         identity = x

#         out = self.conv1(x)
#         out = self.batch_norm1(out)
#         out = self.activ1(out)
#         out = self.conv2(out)
#         out = self.batch_norm2(out)
#         out = self.activ2(out)
#         out = self.conv3(out)
#         out = self.batch_norm3(out)
#         # print(out.shape)
#         # print(identity.shape)
#         if self.downsample is not None:
#             identity = self.downsample(x)

#         out += identity

#         return out

# class Image2BiomassModel(nn.Module):

#     def __init__(self):
#         super(Image2BiomassModel, self).__init__()
#         # self.backbone = Dinov2Model.from_pretrained(
#         #     "/kaggle/working/dinov2/pytorch/base/1/"
#         # )

#         self.backbone = BackBone(num_channels=3)
#         self.prelu = nn.PReLU()

#         # ---- MLP Head ----
#         self.noise = nn.Sequential(
#             nn.AlphaDropout(0.1),
#         )

#         self.fc1 = nn.Sequential(
#             nn.Linear(HEIGHT * WIDTH * NUM_CHANNELS, 512),
#             nn.BatchNorm1d(512),
#             nn.PReLU(),
#             nn.Dropout(0.4),
#         )

#         self.fc2 = nn.Sequential(
#             nn.Linear(512, 256),
#             nn.LayerNorm(256),
#             nn.PReLU(),
#             nn.Linear(256, 128),
#             nn.LayerNorm(128),
#             nn.PReLU(),
#             nn.Dropout(0.4),
#         )

#         self.residual = nn.Sequential(
#             nn.Linear(128, 128),
#             nn.LayerNorm(128),
#             nn.PReLU(),
#             nn.Linear(128, 128),
#             nn.LayerNorm(128),
#         )
#         self.out = nn.Linear(128, 3)

#         self.criterion = nn.SmoothL1Loss(beta=0.5)
        
#     def forward(self, x, y=None):
#         # DINOv2-giant expects normalized images and outputs [B, 1536]
#         outputs = self.backbone(x, )
#         # x = outputs.last_hidden_state[:, 0]

#         x = outputs.view(outputs.shape[0], -1)
#         # print()
#         x = self.noise(x)
#         x = self.fc1(x)
#         x = self.fc2(x)

#         res = self.residual(x)
#         x = x + res
#         x = self.prelu(x)
#         # x = F.Mish(x)

#         preds = self.out(x)

#         loss = None
#         if y is not None:
#             loss = self.criterion(preds, y)

#         return preds, loss

# sample = next(iter(train_dataloader))
# model = Image2BiomassModel()

# model(sample[0], sample[2])

# Train Loop

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Image2BiomassModel().to(device)
BATCH_SIZE=8
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# base_optimizer = torch.optim.AdamW
# optimizer = SAM(model.parameters(), base_optimizer, lr=1e-4, weight_decay=1e-2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
weights = torch.tensor([0.1, 0.1, 0.1, 0.2, 0.5], device=device)

train_losses, val_losses = [], []
train_r2_history, val_r2_history = [], []

Downloading: "https://download.pytorch.org/models/resnet152-f82ba261.pth" to /root/.cache/torch/hub/checkpoints/resnet152-f82ba261.pth


100%|██████████| 230M/230M [00:00<00:00, 255MB/s]  


In [18]:
def weighted_r2(y_true, y_pred, weights):
    y_true = target_untransform(y_true)
    y_pred = target_untransform(y_pred)

    
    # create new columns
    gdm = (y_true[:, 0] + y_true[:, 2]).unsqueeze(1)   # (batch, 1)
    tot = (y_true[:, 0] + y_true[:, 1] + y_true[:, 2]).unsqueeze(1)
    
    gdm_pred = (y_pred[:, 0] + y_pred[:, 2]).unsqueeze(1)   # (batch, 1)
    tot_pred = (y_pred[:, 0] + y_pred[:, 1] + y_pred[:, 2]).unsqueeze(1)

    # append columns
    y_true = torch.cat([y_true, gdm, tot], dim=1)
    y_pred = torch.cat([y_pred, gdm_pred, tot_pred], dim=1)

    # print("Prediction:", y_pred)
    # print("Target:", y_true)

    # compute weighted R2
    mean = y_true.mean(dim=0)
    SSE = ((y_true - y_pred)**2).sum(dim=0)
    TSS = ((y_true - mean)**2).sum(dim=0)
    TSS = torch.clamp(TSS, min=1e-8)
    R2 = 1 - SSE / TSS
    R2 = torch.clamp(R2, min=-10, max=1)
    return (R2 * weights).sum() / weights.sum()

In [19]:
%%capture
!pip install wandb

In [20]:
import wandb
import os
os.environ["WANDB_API_KEY"] = "f5498d8776689da0795dbdee5044ad07e5c956ad"
wandb.login(key=os.environ["WANDB_API_KEY"])

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sayid-10121012 (sayid-10121012-universitas-komputer-indonesia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [21]:
import wandb

#HYPERPARAMETERS
run = wandb.init(
    project="IMAGE2BIOMASSPREDICTION",
    config={
        "learning_rate": 0.02,
        "architecture": "Resnet50",
        "dataset": "Image2Biomass",
        "epochs": 100,
    },
)

wandb.watch(model, log="all", log_freq=100)

In [22]:
from tqdm import tqdm
import torch
from torch.nn.utils import clip_grad_norm_

epochs = 500
for epoch in range(1, epochs+1):
    model.train()
    train_loss = 0
    train_r2_scores = []

    for imgs, _, y in tqdm(train_dataloader, desc=f"[Train] Epoch {epoch}"):

        imgs, y = imgs.to(device), y.to(device)

        # preds, loss = model(imgs, y)
        # optimizer.zero_grad()
        # print(loss.requires_grad)
        # def closure():
        #     # optimizer.zero_grad()
        #     loss.backward()
        #     return loss
        # # loss.backward()
        # optimizer.step(closure)
        
        preds, loss = model(imgs, y)

        # L1 REGULARIZATION
        l1_lambda = 1e-8
        reg_loss = sum(param.abs().sum() for param in model.parameters())
        loss = loss + l1_lambda * reg_loss
        optimizer.zero_grad()
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        train_r2_scores.append(weighted_r2(y, preds, weights).item())

    avg_train_loss = train_loss / len(train_dataloader)
    avg_train_r2 = sum(train_r2_scores) / len(train_r2_scores)

    # VALIDATION
    model.eval()
    val_loss = 0
    val_r2_scores = []

    with torch.no_grad():
        for imgs, _, y in tqdm(val_dataloader, desc=f"[Val] Epoch {epoch}"):
            imgs, y = imgs.to(device), y.to(device)
            preds, loss = model(imgs, y)
            val_loss += loss.item()
            val_r2_scores.append(weighted_r2(y, preds, weights).item())

    avg_val_loss = val_loss / len(val_dataloader)
    avg_val_r2 = sum(val_r2_scores) / len(val_r2_scores)
    val_losses.append(avg_val_loss)
    train_losses.append(avg_train_loss)
    val_r2_history.append(avg_val_r2)
    train_r2_history.append(avg_train_r2)
    wandb.log({
    "epoch": epoch,
    "train_loss": avg_train_loss,
    "train_r2": avg_train_r2,
    "val_loss": avg_val_loss,
    "val_r2": avg_val_r2,
    "lr": optimizer.param_groups[0]["lr"],
    })

    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | "
          f"Train R2: {avg_train_r2:.4f} | Val Loss: {avg_val_loss:.4f} | Val R2: {avg_val_r2:.4f}")

[Val] Epoch 1: 100%|██████████| 9/9 [00:05<00:00,  1.75it/s]


Epoch 1 | Train Loss: 0.8925 | Train R2: -1.2671 | Val Loss: 0.8462 | Val R2: -1.0141


[Val] Epoch 2: 100%|██████████| 9/9 [00:05<00:00,  1.72it/s]


Epoch 2 | Train Loss: 0.7886 | Train R2: -0.5258 | Val Loss: 0.8268 | Val R2: -3.4310


[Val] Epoch 3: 100%|██████████| 9/9 [00:04<00:00,  1.82it/s]


...

Epoch 199 | Train Loss: 0.4938 | Train R2: -0.1027 | Val Loss: 0.5781 | Val R2: -0.1112


[Val] Epoch 200: 100%|██████████| 9/9 [00:05<00:00,  1.76it/s]


Epoch 200 | Train Loss: 0.4924 | Train R2: -0.0880 | Val Loss: 0.5362 | Val R2: -0.0296


[Train] Epoch 201:  17%|█▋        | 6/36 [00:06<00:32,  1.08s/it]


KeyboardInterrupt: 

In [23]:
wandb.finish()

epoch,▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▆▆▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▆▅▄▄▄▄▃▃▃▃▃▃▂▂▃▃▂▂▂▂▂▃▂▁▂▂▂▂▂▂▂▂▁▂▂▁▁▂▂
train_r2,▂▁▃▆▄▅▅▇▄▆▆▆▆▃▆▇▄▇▇▇▆▇▅▄▅▆██▅▇▇▇▆▅▇▇▇▇▇▇
val_loss,▆▇███▇█▅█▇▆▇▅▃▃▂▂▄▃▃▂▂▃▂▂▂▂▂▄▁▂▂▂▁▂▃▁▂▂▂
val_r2,▅▁▁▃▄▅▆▆▅▆▅▇▇▅▄▇▇▇▆▇█▆▆█▆█▇██▇██████▇███
epoch,200
lr,0.0001
train_loss,0.49237
train_r2,-0.088
val_loss,0.53616


In [26]:
torch.save(model.state_dict(), "image2biomass_weights_resnet152_best.pth")
print("Model saved to image2biomass_weights_resnet152_best.pth")

Model saved to image2biomass_weights_resnet152.pth


In [27]:
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm

model = Image2BiomassModel().to(device)
model.load_state_dict(torch.load("image2biomass_weights_resnet152_best.pth", map_location=device))
model.eval()

rows = []

target_cols = [
    "Dry_Green_g",
    "Dry_Dead_g",
    "Dry_Clover_g",
    "GDM_g",
    "Dry_Total_g",
]

test_dataset = Image2BioMassTestDataset(
    dataset_path="/kaggle/input/csiro-biomass/",
    img_transform=val_transform
    # img_transform=image_transform,
)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

with torch.no_grad():
    for imgs, _, sample_ids in tqdm(test_dataloader, desc="Inference"):
        imgs = imgs.to(device)

        # (B, 3) - model outputs: [Dry_Green_g, Dry_Dead_g, Dry_Clover_g]
        y_pred, _ = model(imgs, y=None)
        print("y_pred transformed:", y_pred)

        y_pred = target_untransform(y_pred).cpu().numpy()
        
        print("y_pred pure:", y_pred)
        # extract 3 predictions in the correct order
        dg = y_pred[:, 0]  # Dry_Green_g
        dd = y_pred[:, 1]  # Dry_Dead_g
        dc = y_pred[:, 2]  # Dry_Clover_g

        # compute extra targets
        gdm = dg + dc
        dry_total = dg + dd + dc

        preds5 = np.stack([dg, dd, dc, gdm, dry_total], axis=1)
        np.set_printoptions(suppress=True, precision=4)
        # print(preds5)

        # build submission rows
        for sid, pred_vec in zip(sample_ids, preds5):
            for col, value in zip(target_cols, pred_vec):
                rows.append({
                    "sample_id": f"{sid}__{col}",
                    "target": float(value)
                })

df_submit = pd.DataFrame(rows)
df_submit.to_csv("submission.csv", index=False)
print("Saved submission.csv")
df_submit.head(20)

Inference: 100%|██████████| 1/1 [00:00<00:00,  4.87it/s]

y_pred transformed: tensor([[2.7524, 3.5018, 0.3407]])
y_pred pure: [[14.679949   32.176727    0.40594733]]
Saved submission.csv


,sample_id,target
0,ID1001187975__Dry_Green_g,14.679949
1,ID1001187975__Dry_Dead_g,32.176727
2,ID1001187975__Dry_Clover_g,0.405947
3,ID1001187975__GDM_g,15.085896
4,ID1001187975__Dry_Total_g,47.262623


# Download Checkpoint

In [30]:
!find /kaggle -name "image2biomass_weights_resnet152_best.pth"

/kaggle/working/image2biomass_weights_resnet152_best.pth


In [31]:
import shutil
from IPython.display import FileLink

shutil.make_archive("model_weights", "zip", "/kaggle/working", "image2biomass_weights_resnet152_best.pth")
    FileLink("model_weights.zip")

/kaggle/working/model_weights.zip

In [34]:
import torch

torch.cuda.is_available()

False